# Estimasi SOC CALCE: MLP, GRU, LSTM

Notebook penelitian komputasional. Satu sel SP20-2, empat profil, tiga suhu lingkungan, dua tingkat SOC awal nominal. Acuan SOC direkonstruksi dari kapasitas penguji dan kapasitas kalibrasi terpisah; bukan pengukuran SOC independen. NASA tidak dimasukkan sebelum audit terpisah.

**Penting:** sebagian acuan melewati nol pada akhir pengosongan. Nilainya dipertahankan untuk audit dan jendela di luar [0,1] dikeluarkan secara seragam, tanpa clipping target. Pembagian rekaman dilakukan sebelum pembentukan jendela. Hasil pilot adalah validasi, bukan hasil uji akhir. Tidak ada klaim arsitektur baru atau penerapan BMS.

Jalankan sel secara berurutan. Eksperimen utama dapat memerlukan beberapa menit atau lebih pada CPU. Script sengaja menggunakan CPU agar pembandingan waktu berada pada perangkat yang sama; waktu tidak mewakili ESP32.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.26,<3', 'pandas>=2.2,<4', 'openpyxl>=3.1,<4', 'xlrd==2.0.2', 'matplotlib>=3.8,<4', 'torch>=2.4,<3'])


In [ ]:
from pathlib import Path
import json, sys, subprocess
ROOT = Path('/content/soc_calce') if Path('/content').exists() else Path.cwd()/'soc_calce_notebook_run'
ROOT.mkdir(parents=True, exist_ok=True)
embedded = {'scripts/fetch_data.py': '"""Download original CALCE archives; retain hashes and source URLs."""\nfrom pathlib import Path\nimport argparse, hashlib, json, urllib.request, zipfile\n\nROOT = Path(__file__).resolve().parents[1]\nBASE = \'https://web.calce.umd.edu/batteries/data/\'\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--temperatures\', nargs=\'+\', type=int, default=[25])\n    args = parser.parse_args()\n    raw = ROOT / \'data\' / \'raw\'\n    raw.mkdir(parents=True, exist_ok=True)\n    names = [\'SP2_Initial_capacity_10_16_2015.zip\']\n    names += [f\'SP2_{t}C_{p}.zip\' for t in args.temperatures for p in [\'DST\',\'FUDS\',\'US06\',\'BJDST\']]\n    manifest_path = raw / \'download_manifest.json\'\n    manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}\n    for name in names:\n        target = raw / name\n        url = BASE + name\n        if not target.exists():\n            print(\'Downloading\', name, flush=True)\n            req = urllib.request.Request(url, headers={\'User-Agent\': \'Academic SOC research; Python urllib\'})\n            temp = target.with_suffix(\'.part\')\n            with urllib.request.urlopen(req, timeout=90) as response, temp.open(\'wb\') as output:\n                while chunk := response.read(1024 * 1024):\n                    output.write(chunk)\n            if not zipfile.is_zipfile(temp):\n                raise ValueError(f\'Not a ZIP archive: {name}\')\n            temp.replace(target)\n        with zipfile.ZipFile(target) as archive:\n            members = archive.namelist()\n        digest = hashlib.sha256(target.read_bytes()).hexdigest()\n        if name in manifest and manifest[name][\'sha256\'] != digest:\n            raise ValueError(f\'Source/cache hash changed for {name}; review provenance before replacing the manifest\')\n        manifest[name] = dict(url=url, bytes=target.stat().st_size,\n            sha256=digest, members=members)\n        manifest_path.write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n        print(name, target.stat().st_size, \'bytes;\', len(members), \'members\', flush=True)\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/inspect_data.py': '"""Inspect original CALCE Arbin files without assuming a SOC column exists."""\nfrom pathlib import Path\nimport sys, zipfile, json, collections, io\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT / \'.deps\'))\nimport xlrd\nimport openpyxl\n\ndef workbook_tables(blob):\n    if blob[:2] == b\'PK\':\n        book = openpyxl.load_workbook(io.BytesIO(blob), read_only=True, data_only=True)\n        for sheet in book:\n            yield sheet.title, list(sheet.values)\n        book.close()\n    else:\n        book = xlrd.open_workbook(file_contents=blob)\n        for sheet in book.sheets():\n            yield sheet.name, [sheet.row_values(i) for i in range(sheet.nrows)]\n\ndef inspect(path):\n    output = []\n    with zipfile.ZipFile(path) as archive:\n        for name in archive.namelist():\n            if not name.lower().endswith(\'.xls\'):\n                continue\n            record = dict(archive=path.name, file=name, sheets=[])\n            for sheet_name, table in workbook_tables(archive.read(name)):\n                info = dict(name=sheet_name, rows=len(table), columns=len(table[0]) if table else 0,\n                            first_rows=table[:3])\n                head = table[0] if table else []\n                if \'Step_Index\' in head and \'Current(A)\' in head:\n                    index = {h:i for i,h in enumerate(head)}\n                    groups = collections.defaultdict(list)\n                    for row in table[1:]:\n                        groups[row[index[\'Step_Index\']]].append(row)\n                    info[\'step_summary\'] = []\n                    for step,rows in groups.items():\n                        entry = dict(step=step, n=len(rows))\n                        for key in [\'Test_Time(s)\',\'Current(A)\',\'Voltage(V)\',\'Charge_Capacity(Ah)\',\'Discharge_Capacity(Ah)\']:\n                            if key in index:\n                                vals = [r[index[key]] for r in rows]\n                                entry[key] = dict(first=vals[0], last=vals[-1], min=min(vals), max=max(vals))\n                        info[\'step_summary\'].append(entry)\n                record[\'sheets\'].append(info)\n            output.append(record)\n    return output\n\nif __name__ == \'__main__\':\n    result = []\n    for path in sorted((ROOT/\'data/raw\').glob(\'*.zip\')):\n        print(\'Inspecting\', path.name, flush=True)\n        result.extend(inspect(path))\n    out = ROOT/\'results/data_inspection.json\'\n    out.parent.mkdir(parents=True, exist_ok=True)\n    out.write_text(json.dumps(result, indent=2, default=str), encoding=\'utf-8\')\n    for record in result:\n        print(record[\'file\'])\n        for sheet in record[\'sheets\']:\n            print(\' \', sheet[\'name\'], sheet[\'rows\'], \'rows;\', sheet[\'first_rows\'][:1])\n            for step in sheet.get(\'step_summary\',[]):\n                print(\'  \', json.dumps(step))\n', 'scripts/prepare_data.py': '"""Causal 1 s sampling and explicitly reconstructed charge-counter SOC reference."""\nfrom pathlib import Path\nimport re, zipfile, json, hashlib\nimport numpy as np\nimport pandas as pd\nfrom inspect_data import workbook_tables\n\nROOT = Path(__file__).resolve().parents[1]\n\ndef load_channel(blob):\n    channels = []\n    for name, table in workbook_tables(blob):\n        if table and \'Current(A)\' in table[0] and \'Test_Time(s)\' in table[0]:\n            df = pd.DataFrame(table[1:], columns=table[0])\n            channels.append(df)\n    if len(channels) != 1:\n        raise ValueError(f\'Expected exactly one measurement channel, got {len(channels)}\')\n    return channels[0]\n\ndef main():\n    raw, processed = ROOT/\'data/raw\', ROOT/\'data/processed\'\n    processed.mkdir(parents=True, exist_ok=True)\n    capacity_archive = raw/\'SP2_Initial_capacity_10_16_2015.zip\'\n    with zipfile.ZipFile(capacity_archive) as archive:\n        blob = archive.read(archive.namelist()[0])\n    capacity_df = load_channel(blob)\n    # Step 6 is the first full 1 A discharge in the separate capacity-calibration run.\n    full_discharge = capacity_df[capacity_df[\'Step_Index\'] == 6]\n    before = capacity_df[capacity_df[\'Step_Index\'] < 6][\'Discharge_Capacity(Ah)\'].iloc[-1]\n    q_ref = float(full_discharge[\'Discharge_Capacity(Ah)\'].iloc[-1] - before)\n    if not 1.5 < q_ref < 2.5:\n        raise ValueError(f\'Unexpected calibration capacity: {q_ref}\')\n    inventory = []\n    for path in sorted(raw.glob(\'SP2_*C_*.zip\')):\n        match = re.fullmatch(r\'SP2_(\\d+)C_(DST|FUDS|US06|BJDST)\\.zip\', path.name)\n        if not match:\n            continue\n        temp, profile = int(match[1]), match[2]\n        with zipfile.ZipFile(path) as archive:\n            for filename in archive.namelist():\n                if not filename.lower().endswith(\'.xls\'):\n                    continue\n                initial_level = int(re.search(r\'_(50|80)SOC\', filename)[1])\n                df = load_channel(archive.read(filename))\n                df = df.sort_values(\'Test_Time(s)\', kind=\'stable\').reset_index(drop=True)\n                if df[\'Test_Time(s)\'].duplicated().any():\n                    raise ValueError(f\'Duplicate time in {filename}\')\n                # These step meanings were verified in the original 25 C files.\n                # Require supporting charge/current checks for every new record.\n                full = df[df[\'Step_Index\'] == 3]\n                dynamic_idx = df.index[df[\'Step_Index\'] == 7]\n                if full.empty or dynamic_idx.empty or full[\'Voltage(V)\'].iloc[-1] < 4.15:\n                    raise ValueError(f\'Unrecognized conditioning steps: {filename}\')\n                c_full = float(full[\'Charge_Capacity(Ah)\'].iloc[-1])\n                d_full = float(full[\'Discharge_Capacity(Ah)\'].iloc[-1])\n                dynamic = df.loc[dynamic_idx[0]:dynamic_idx[-1]].copy()\n                t = dynamic[\'Test_Time(s)\'].to_numpy(float)\n                v = dynamic[\'Voltage(V)\'].to_numpy(float)\n                i = dynamic[\'Current(A)\'].to_numpy(float)  # source sign: positive charge\n                c = dynamic[\'Charge_Capacity(Ah)\'].to_numpy(float)\n                d = dynamic[\'Discharge_Capacity(Ah)\'].to_numpy(float)\n                soc = 1 + ((c - c_full) - (d - d_full)) / q_ref\n                if np.any(np.diff(c) < -1e-7) or np.any(np.diff(d) < -1e-7):\n                    raise ValueError(f\'Capacity counters reset: {filename}\')\n                if not all(np.isfinite(x).all() for x in [t,v,i,soc]):\n                    raise ValueError(f\'Non-finite data: {filename}\')\n                # Causal zero-order hold. No centered interpolation or future samples.\n                grid = t[0] + np.arange(int(np.floor(t[-1]-t[0])) + 1)\n                pos = np.searchsorted(t, grid, side=\'right\') - 1\n                x = np.column_stack([v[pos], i[pos], np.full(len(pos), temp)])\n                target = soc[pos]\n                record_id = f\'{profile}_{temp}C_{initial_level}SOC\'\n                segment = np.zeros(len(grid), dtype=int)\n                # Do not build windows across intervals unsupported by a recent sample.\n                valid = (grid - t[pos]) <= 5.0\n                segment[1:] = np.cumsum((~valid[1:]) | (~valid[:-1]))\n                np.savez_compressed(processed/f\'{record_id}.npz\', x=x.astype(\'float32\'),\n                    soc=target.astype(\'float32\'), t=(grid-t[0]).astype(\'float32\'),\n                    valid=valid, segment=segment)\n                outlier_count = int(((target < 0) | (target > 1)).sum())\n                inventory.append(dict(record_id=record_id, profile=profile, temperature_c=temp,\n                    initial_soc_label=initial_level, initial_soc_reference=float(target[0]),\n                    final_soc_reference=float(target[-1]), n_samples=len(grid),\n                    valid_samples=int(valid.sum()), max_raw_gap_s=float(np.diff(t).max()),\n                    outside_0_1_samples=outlier_count, q_reference_ah=q_ref,\n                    source_archive=path.name, source_member=filename,\n                    source_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n                    soc_reference=\'reconstructed from tester charge/discharge counters; fixed capacity from independent initial-capacity run\',\n                    temperature_feature=\'ambient chamber setting; not measured cell temperature\'))\n                print(record_id, len(grid), \'samples; SOC\', round(target[0]*100,3),\n                      \'to\', round(target[-1]*100,3), \'%; out of range\', outlier_count, flush=True)\n    report = dict(q_reference_ah=q_ref, reference_capacity_scope=\'single initial capacity test at reference conditions; not temperature-adaptive\',\n        soc_formula=\'1 + ((charge_Ah-charge_at_full_Ah) - (discharge_Ah-discharge_at_full_Ah))/Q_reference_Ah\',\n        assumptions=[\'end of CCCV step 3 is full charge\', \'coulombic efficiency is one\',\n                     \'capacity reference stays constant across profiles and temperatures\'],\n        target_clipping=False, sampling=\'1 s causal last-observation hold; samples older than 5 s excluded\',\n        records=inventory)\n    (processed/\'inventory.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    pd.DataFrame(inventory).to_csv(ROOT/\'results/data_inventory.csv\', index=False)\n\nif __name__ == \'__main__\':\n    main()\n', 'scripts/train.py': '"""Reproducible MLP/GRU/LSTM comparisons with record-level data separation."""\nfrom pathlib import Path\nimport argparse, copy, json, random, time, platform\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\n\nROOT = Path(__file__).resolve().parents[1]\n\nclass Windows(Dataset):\n    def __init__(self, records, mean, std, window, stride):\n        self.window, self.arrays, self.locations = window, [], []\n        for record in records:\n            with np.load(ROOT/\'data/processed\'/f"{record[\'record_id\']}.npz") as d:\n                x = ((d[\'x\'] - mean) / std).astype(\'float32\')\n                y, valid, segment, times = d[\'soc\'], d[\'valid\'], d[\'segment\'], d[\'t\']\n                valid = valid & (y >= 0) & (y <= 1)\n            idx = len(self.arrays)\n            self.arrays.append((torch.from_numpy(x), torch.from_numpy(y.copy()), times.copy(), record))\n            for end in range(window-1, len(x), stride):\n                start = end-window+1\n                if valid[start:end+1].all() and segment[start] == segment[end]:\n                    self.locations.append((idx, end))\n        if not self.locations:\n            raise ValueError(\'No valid windows\')\n    def __len__(self):\n        return len(self.locations)\n    def __getitem__(self, index):\n        rec, end = self.locations[index]\n        x,y,_,_ = self.arrays[rec]\n        return x[end-self.window+1:end+1], y[end]\n\nclass Estimator(nn.Module):\n    def __init__(self, kind, window, hidden):\n        super().__init__()\n        self.kind = kind\n        if kind == \'MLP\':\n            self.core = nn.Sequential(nn.Flatten(), nn.Linear(window*3,hidden), nn.Tanh())\n        else:\n            cls = nn.GRU if kind == \'GRU\' else nn.LSTM\n            self.core = cls(3,hidden,batch_first=True)\n        self.head = nn.Sequential(nn.Linear(hidden,1), nn.Sigmoid())\n    def forward(self,x):\n        z = self.core(x)\n        if self.kind != \'MLP\':\n            z = z[0][:,-1,:]\n        return self.head(z).squeeze(-1)\n\ndef predict(model, dataset, batch):\n    result = []\n    model.eval()\n    with torch.inference_mode():\n        for x,_ in DataLoader(dataset,batch_size=batch,shuffle=False):\n            result.append(model(x).numpy())\n    return np.concatenate(result)\n\ndef score(y,p):\n    e = (p-y)*100\n    return dict(MAE_pp=float(np.mean(np.abs(e))), RMSE_pp=float(np.sqrt(np.mean(e**2))),\n                max_abs_error_pp=float(np.max(np.abs(e))))\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--mode\',choices=[\'pilot\',\'full\',\'cross-temperature\'],default=\'pilot\')\n    parser.add_argument(\'--epochs\',type=int)\n    parser.add_argument(\'--seeds\',type=int,nargs=\'+\')\n    args=parser.parse_args()\n    config=json.loads((ROOT/\'protocol_v1.json\').read_text())\n    inventory=json.loads((ROOT/\'data/processed/inventory.json\').read_text())\n    records=inventory[\'records\']\n    temperatures=[25] if args.mode==\'pilot\' else [0,25,45]\n    train_temps=[0,45] if args.mode==\'cross-temperature\' else temperatures\n    test_temps=[25] if args.mode==\'cross-temperature\' else temperatures\n    train_r=[r for r in records if r[\'profile\']==\'DST\' and r[\'temperature_c\'] in train_temps]\n    val_r=[r for r in records if r[\'profile\']==\'FUDS\' and r[\'temperature_c\'] in train_temps]\n    test_r=[] if args.mode==\'pilot\' else [r for r in records if r[\'profile\'] in [\'US06\',\'BJDST\'] and r[\'temperature_c\'] in test_temps]\n    if len(train_r)!=2*len(train_temps) or len(val_r)!=2*len(train_temps) or (args.mode!=\'pilot\' and len(test_r)!=4*len(test_temps)):\n        raise ValueError(\'Required records missing; finish download and preparation first\')\n    train_ids={r[\'record_id\'] for r in train_r}; val_ids={r[\'record_id\'] for r in val_r}; test_ids={r[\'record_id\'] for r in test_r}\n    assert not (train_ids & val_ids or train_ids & test_ids or val_ids & test_ids)\n    pooled=[]\n    for record in train_r:\n        with np.load(ROOT/\'data/processed\'/f"{record[\'record_id\']}.npz") as d:\n            mask=d[\'valid\'] & (d[\'soc\']>=0) & (d[\'soc\']<=1)\n            pooled.append(d[\'x\'][mask])\n    pooled=np.concatenate(pooled)\n    mean=pooled.mean(0); std=pooled.std(0); std[std<1e-6]=1\n    window=config[\'window_samples\']; batch=config[\'batch_size\']\n    train=Windows(train_r,mean,std,window,config[\'train_stride\'])\n    val=Windows(val_r,mean,std,window,1)\n    test=Windows(test_r,mean,std,window,1) if test_r else None\n    seeds=args.seeds or ([11] if args.mode==\'pilot\' else config[\'seeds\'])\n    epochs=args.epochs or (5 if args.mode==\'pilot\' else config[\'epochs_max\'])\n    out=ROOT/\'results\'/args.mode; out.mkdir(parents=True,exist_ok=True)\n    torch.set_num_threads(2)\n    torch.use_deterministic_algorithms(True)\n    run_info=dict(mode=args.mode,epochs_max=epochs,seeds=seeds,train_records=sorted(train_ids),\n        validation_records=sorted(val_ids),test_records=sorted(test_ids),normalization_mean=mean.tolist(),\n        normalization_std=std.tolist(),window=window,train_windows=len(train),validation_windows=len(val),\n        test_windows=len(test) if test else 0,torch=torch.__version__,python=platform.python_version(),\n        platform=platform.platform(),test_evaluated=bool(test),reference=\'reconstructed SOC; not independently measured\')\n    (out/\'run_info.json\').write_text(json.dumps(run_info,indent=2),encoding=\'utf-8\')\n    metrics=[]; predictions=[]; histories=[]\n    for seed in seeds:\n        for kind in config[\'models\']:\n            random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n            generator=torch.Generator().manual_seed(seed)\n            loader=DataLoader(train,batch_size=batch,shuffle=True,generator=generator)\n            model=Estimator(kind,window,config[\'hidden_units\'])\n            optimizer=torch.optim.Adam(model.parameters(),lr=config[\'learning_rate\'])\n            loss_fn=nn.MSELoss(); best=float(\'inf\'); best_state=None; stale=0; best_epoch=0\n            started=time.perf_counter()\n            for epoch in range(1,epochs+1):\n                model.train(); losses=[]\n                for x,y in loader:\n                    optimizer.zero_grad(); loss=loss_fn(model(x),y); loss.backward()\n                    nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step(); losses.append(loss.item())\n                p=predict(model,val,batch)\n                y=np.array([float(val.arrays[r][1][e]) for r,e in val.locations])\n                vloss=float(np.mean((p-y)**2))\n                histories.append(dict(model=kind,seed=seed,epoch=epoch,train_MSE=float(np.mean(losses)),validation_MSE=vloss))\n                if vloss<best:\n                    best=vloss; best_state=copy.deepcopy(model.state_dict()); best_epoch=epoch; stale=0\n                else:\n                    stale+=1\n                if epoch==1 or epoch%5==0:\n                    print(args.mode,kind,seed,\'epoch\',epoch,\'validation RMSE pp\',round(np.sqrt(vloss)*100,4),flush=True)\n                if stale>=config[\'early_stopping_patience\']:\n                    break\n            training_s=time.perf_counter()-started\n            model.load_state_dict(best_state)\n            torch.save(dict(state_dict=best_state,kind=kind,window=window,hidden=config[\'hidden_units\'],\n                mean=mean.tolist(),std=std.tolist(),seed=seed),out/f\'{kind}_seed{seed}.pt\')\n            for split,dataset in [(\'validation\',val)]+([(\'test\',test)] if test else []):\n                p=predict(model,dataset,batch)\n                # Warmed, batched CPU throughput, not embedded or single-sample latency.\n                started=time.perf_counter(); predict(model,dataset,batch)\n                infer_ms=(time.perf_counter()-started)*1000/len(dataset)\n                frame=[]\n                for k,(rec,end) in enumerate(dataset.locations):\n                    _,labels,times,info=dataset.arrays[rec]\n                    frame.append(dict(model=kind,seed=seed,split=split,record_id=info[\'record_id\'],\n                        profile=info[\'profile\'],temperature_c=info[\'temperature_c\'],t_s=float(times[end]),\n                        SOC_reference=float(labels[end]),SOC_estimate=float(p[k])))\n                frame=pd.DataFrame(frame); predictions.append(frame)\n                for rid,g in frame.groupby(\'record_id\'):\n                    row=dict(model=kind,seed=seed,split=split,record_id=rid,n=len(g),best_epoch=best_epoch,\n                        training_seconds=training_s,parameters=sum(p.numel() for p in model.parameters()),\n                        batch_inference_ms_per_window=infer_ms)\n                    row.update(score(g.SOC_reference.to_numpy(),g.SOC_estimate.to_numpy())); metrics.append(row)\n                print(kind,seed,split,\'macro RMSE pp\',round(np.mean([r[\'RMSE_pp\'] for r in metrics if r[\'model\']==kind and r[\'seed\']==seed and r[\'split\']==split]),4),flush=True)\n            pd.DataFrame(metrics).to_csv(out/\'metrics_per_record.csv\',index=False)\n            pd.DataFrame(histories).to_csv(out/\'history.csv\',index=False)\n            pd.concat(predictions,ignore_index=True).to_csv(out/\'predictions.csv\',index=False)\n    m=pd.DataFrame(metrics)\n    macro=m.groupby([\'model\',\'seed\',\'split\'],as_index=False)[[\'MAE_pp\',\'RMSE_pp\',\'max_abs_error_pp\']].mean()\n    macro.to_csv(out/\'macro_per_seed.csv\',index=False)\n    macro.groupby([\'model\',\'split\'])[[\'MAE_pp\',\'RMSE_pp\']].agg([\'mean\',\'std\']).to_csv(out/\'summary.csv\')\n    print(\'Saved\',out,flush=True)\n\nif __name__==\'__main__\':\n    main()\n', 'scripts/validate_pipeline.py': '"""Audit data provenance and split/window invariants before interpreting metrics."""\nfrom pathlib import Path\nimport hashlib, json\nimport numpy as np\nimport pandas as pd\nfrom train import Windows\n\nROOT = Path(__file__).resolve().parents[1]\n\ndef main():\n    downloads=json.loads((ROOT/\'data/raw/download_manifest.json\').read_text())\n    inv=json.loads((ROOT/\'data/processed/inventory.json\').read_text())\n    assert len(downloads)==13, \'Expected 12 dynamic archives plus calibration\'\n    for name,meta in downloads.items():\n        assert hashlib.sha256((ROOT/\'data/raw\'/name).read_bytes()).hexdigest()==meta[\'sha256\']\n    records=inv[\'records\']\n    assert len(records)==24\n    assert len({r[\'record_id\'] for r in records})==24\n    assert abs(inv[\'q_reference_ah\']-2.028344937765965)<1e-10\n    excluded=0; total=0; balance=[]\n    for r in records:\n        with np.load(ROOT/\'data/processed\'/f"{r[\'record_id\']}.npz") as d:\n            assert d[\'x\'].shape==(r[\'n_samples\'],3)\n            assert np.allclose(np.diff(d[\'t\']),1)\n            assert np.allclose(d[\'x\'][:,2],r[\'temperature_c\'])\n            assert np.isfinite(d[\'x\']).all() and np.isfinite(d[\'soc\']).all()\n            excluded+=int(((d[\'soc\']<0)|(d[\'soc\']>1)).sum()); total+=len(d[\'soc\'])\n            currents=d[\'x\'][:,1].astype(float)\n            integral=np.sum((currents[1:]+currents[:-1])*0.5*np.diff(d[\'t\'].astype(float)))/3600\n            counter_delta=(float(d[\'soc\'][-1])-float(d[\'soc\'][0]))*inv[\'q_reference_ah\']\n            balance.append(dict(record_id=r[\'record_id\'],net_charge_integral_Ah=integral,\n                net_charge_counter_Ah=counter_delta,\n                difference_SOC_pp=(integral-counter_delta)/inv[\'q_reference_ah\']*100))\n    # A deliberately invalid label in the middle of a window must prevent its use.\n    sets={p:{r[\'record_id\'] for r in records if r[\'profile\']==p} for p in [\'DST\',\'FUDS\',\'US06\',\'BJDST\']}\n    assert not (sets[\'DST\'] & sets[\'FUDS\'] or sets[\'DST\'] & (sets[\'US06\']|sets[\'BJDST\']))\n    train=[r for r in records if r[\'profile\']==\'DST\']\n    ds=Windows(train,np.zeros(3),np.ones(3),60,5)\n    for rec,end in ds.locations:\n        _,y,_,_=ds.arrays[rec]\n        labels=y[end-59:end+1].numpy()\n        assert (labels>=0).all() and (labels<=1).all()\n    checks=dict(status=\'passed\',archives_verified=len(downloads),records_verified=len(records),\n        raw_reference_samples=total,outside_reference_range_samples=excluded,\n        outside_reference_range_percent=100*excluded/total,\n        Q_reference_Ah=inv[\'q_reference_ah\'],feature_count=3,\n        target_clipping=False,record_level_split_disjoint=True,train_windows=len(ds),\n        max_abs_current_balance_difference_pp=max(abs(r[\'difference_SOC_pp\']) for r in balance),\n        note=\'Pipeline checks do not establish independent SOC truth or scientific novelty\')\n    (ROOT/\'results/pipeline_validation.json\').write_text(json.dumps(checks,indent=2))\n    pd.DataFrame(balance).to_csv(ROOT/\'results/current_balance_check.csv\',index=False)\n    print(json.dumps(checks,indent=2))\n\nif __name__==\'__main__\':\n    main()\n', 'scripts/plot_results.py': '"""Create standalone scientific figures from actual recorded results."""\nfrom pathlib import Path\nimport argparse, json, os\nROOT=Path(__file__).resolve().parents[1]\nos.environ.setdefault(\'MPLCONFIGDIR\',str(ROOT/\'results/.matplotlib\'))\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\ndef main():\n    parser=argparse.ArgumentParser(); parser.add_argument(\'--mode\',default=\'pilot\')\n    args=parser.parse_args(); folder=ROOT/\'results\'/args.mode\n    figures=folder/\'figures\'; figures.mkdir(exist_ok=True)\n    m=pd.read_csv(folder/\'macro_per_seed.csv\')\n    split=\'validation\' if args.mode==\'pilot\' else \'test\'\n    s=m[m.split==split].groupby(\'model\')[\'RMSE_pp\'].agg([\'mean\',\'std\']).reindex([\'MLP\',\'GRU\',\'LSTM\'])\n    fig,ax=plt.subplots(figsize=(6.5,4))\n    ax.bar(s.index,s[\'mean\'],yerr=s[\'std\'].fillna(0),color=[\'#8a9baa\',\'#1a7891\',\'#e49a38\'],capsize=4)\n    ax.set_ylabel(\'Macro RMSE (SOC percentage points)\')\n    ax.set_title(\'Pilot validation at 25 °C\' if args.mode==\'pilot\' else \'SOC estimation on held-out load profiles\')\n    ax.spines[[\'top\',\'right\']].set_visible(False); ax.grid(axis=\'y\',alpha=.2); ax.set_axisbelow(True)\n    fig.tight_layout(); fig.savefig(figures/\'rmse_comparison.png\',dpi=300); fig.savefig(figures/\'rmse_comparison.pdf\'); plt.close(fig)\n    pred=pd.read_csv(folder/\'predictions.csv\')\n    record=\'FUDS_25C_80SOC\' if args.mode==\'pilot\' else \'US06_25C_80SOC\'\n    seed=int(pred.seed.min()); p=pred[(pred.record_id==record)&(pred.seed==seed)&(pred.split==split)]\n    fig,axes=plt.subplots(2,1,figsize=(8,6),sharex=True,gridspec_kw={\'height_ratios\':[2,1]})\n    ref=p[p.model==\'GRU\']\n    axes[0].plot(ref.t_s/60,ref.SOC_reference*100,\'k--\',label=\'Reconstructed reference\',lw=1.5)\n    for kind,color in [(\'MLP\',\'#8a9baa\'),(\'GRU\',\'#1a7891\'),(\'LSTM\',\'#e49a38\')]:\n        q=p[p.model==kind]\n        axes[0].plot(q.t_s/60,q.SOC_estimate*100,label=kind,color=color,alpha=.85,lw=.9)\n        axes[1].plot(q.t_s/60,(q.SOC_estimate-q.SOC_reference)*100,color=color,lw=.7)\n    axes[0].set_ylabel(\'SOC (%)\'); axes[1].set_ylabel(\'Error (pp)\'); axes[1].set_xlabel(\'Elapsed dynamic-profile time (min)\')\n    axes[0].set_title(f"{\'FUDS\' if args.mode==\'pilot\' else \'US06\'}, 25 °C, nominal initial SOC 80%"); axes[0].legend(ncols=2,fontsize=8)\n    for ax in axes:\n        ax.grid(alpha=.2); ax.spines[[\'top\',\'right\']].set_visible(False)\n    fig.tight_layout(); fig.savefig(figures/\'soc_trace.png\',dpi=300); fig.savefig(figures/\'soc_trace.pdf\'); plt.close(fig)\n    inventory=json.loads((ROOT/\'data/processed/inventory.json\').read_text())\n    fig,ax=plt.subplots(figsize=(7,4))\n    for temp,color in [(0,\'#5371a0\'),(25,\'#1a7891\'),(45,\'#e49a38\')]:\n        with np.load(ROOT/\'data/processed\'/f\'DST_{temp}C_80SOC.npz\') as d:\n            ax.plot(d[\'t\']/60,d[\'soc\']*100,label=f\'{temp} °C\',color=color,lw=1)\n    ax.axhline(0,color=\'black\',ls=\'--\',lw=.8)\n    ax.set(xlabel=\'Elapsed dynamic-profile time (min)\',ylabel=\'Reconstructed SOC (%)\',title=\'Reference audit: DST, nominal initial SOC 80%\')\n    ax.legend(title=\'Ambient setting\'); ax.grid(alpha=.2); ax.spines[[\'top\',\'right\']].set_visible(False)\n    fig.tight_layout(); fig.savefig(figures/\'reference_audit.png\',dpi=300); fig.savefig(figures/\'reference_audit.pdf\'); plt.close(fig)\n    if args.mode!=\'pilot\':\n        metrics=pd.read_csv(folder/\'metrics_per_record.csv\')\n        info=pd.DataFrame(inventory[\'records\'])[[\'record_id\',\'temperature_c\',\'profile\']]\n        metrics=metrics[metrics.split==\'test\'].merge(info,on=\'record_id\',validate=\'many_to_one\')\n        by_seed=metrics.groupby([\'model\',\'seed\',\'temperature_c\'],as_index=False).RMSE_pp.mean()\n        grouped=by_seed.groupby([\'model\',\'temperature_c\']).RMSE_pp.agg([\'mean\',\'std\'])\n        fig,ax=plt.subplots(figsize=(7,4)); temps=sorted(metrics.temperature_c.unique()); positions=np.arange(len(temps)); width=.24\n        for offset,(kind,color) in enumerate([(\'MLP\',\'#8a9baa\'),(\'GRU\',\'#1a7891\'),(\'LSTM\',\'#e49a38\')]):\n            group=grouped.loc[kind].reindex(temps)\n            ax.bar(positions+(offset-1)*width,group[\'mean\'],width,yerr=group[\'std\'].fillna(0),label=kind,color=color,capsize=3)\n        ax.set(xticks=positions,xticklabels=[str(t) for t in temps],ylabel=\'Macro RMSE (SOC percentage points)\',xlabel=\'Ambient chamber setting (°C)\',title=\'Held-out load profiles: US06 and BJDST\')\n        ax.legend(); ax.grid(axis=\'y\',alpha=.2); ax.set_axisbelow(True); ax.spines[[\'top\',\'right\']].set_visible(False)\n        fig.tight_layout(); fig.savefig(figures/\'rmse_by_temperature.png\',dpi=300); fig.savefig(figures/\'rmse_by_temperature.pdf\'); plt.close(fig)\n    print(figures)\n\nif __name__==\'__main__\':\n    main()\n', 'scripts/audit_results.py': '"""Independently recompute saved errors and verify matched evaluation targets."""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nimport pandas as pd\nROOT=Path(__file__).resolve().parents[1]\n\ndef main():\n    folder=ROOT/\'results/full\'\n    m=pd.read_csv(folder/\'metrics_per_record.csv\')\n    p=pd.read_csv(folder/\'predictions.csv\')\n    expected={(model,seed) for model in [\'MLP\',\'GRU\',\'LSTM\'] for seed in [11,22,33]}\n    assert set(zip(m.model,m.seed))==expected\n    assert (m[m.split==\'test\'].groupby([\'model\',\'seed\']).size()==12).all()\n    keys=[\'model\',\'seed\',\'split\',\'record_id\']\n    recomputed=[]\n    for key,g in p.groupby(keys):\n        e=(g.SOC_estimate.to_numpy()-g.SOC_reference.to_numpy())*100\n        recomputed.append(dict(zip(keys,key),n=len(g),MAE_pp=np.abs(e).mean(),\n            RMSE_pp=np.sqrt((e**2).mean()),max_abs_error_pp=np.abs(e).max()))\n    check=pd.DataFrame(recomputed).merge(m,on=keys,validate=\'one_to_one\',suffixes=(\'_recomputed\',\'_saved\'))\n    assert len(check)==162\n    worst=0.\n    for metric in [\'MAE_pp\',\'RMSE_pp\',\'max_abs_error_pp\']:\n        difference=float(np.abs(check[metric+\'_recomputed\']-check[metric+\'_saved\']).max())\n        worst=max(worst,difference); assert difference<1e-5\n    for (split,rid),g in p.groupby([\'split\',\'record_id\']):\n        template=None\n        for _,h in g.groupby([\'model\',\'seed\']):\n            values=h[[\'t_s\',\'SOC_reference\']].to_numpy()\n            if template is None: template=values\n            else: assert np.array_equal(values,template)\n    info=dict(status=\'passed\',runs=9,metrics_groups=len(check),prediction_rows=len(p),\n        largest_metric_recomputation_difference_pp=worst,matched_target_times=True,\n        targets_clipped=False,test_tuned=False,\n        protocol_sha256=hashlib.sha256((ROOT/\'protocol_v1.json\').read_bytes()).hexdigest(),\n        train_script_sha256=hashlib.sha256((ROOT/\'scripts/train.py\').read_bytes()).hexdigest(),\n        prediction_csv_sha256=hashlib.sha256((folder/\'predictions.csv\').read_bytes()).hexdigest(),\n        limitation=\'This audit verifies numerical consistency, not independent SOC truth or manuscript novelty\')\n    (folder/\'results_audit.json\').write_text(json.dumps(info,indent=2),encoding=\'utf-8\')\n    print(json.dumps(info,indent=2))\n\nif __name__==\'__main__\':\n    main()\n', 'scripts/reference_sensitivity.py': '"""Exploratory reference-definition sensitivity; no retraining or test-based tuning.\n\nAlternative capacities are scenarios, NOT measured tolerances or corrected truth.\nAll scenarios use the same common evaluation timestamps for a fair comparison.\n"""\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nROOT=Path(__file__).resolve().parents[1]\n\ndef main():\n    folder=ROOT/\'results/full\'\n    p=pd.read_csv(folder/\'predictions.csv\')\n    p=p[p.split==\'test\'].copy()\n    inv=json.loads((ROOT/\'data/processed/inventory.json\').read_text())\n    qref=inv[\'q_reference_ah\']\n    capacities=[2.0,qref,2.06]\n    labels=np.column_stack([1-(1-p.SOC_reference.to_numpy())*qref/q for q in capacities])\n    valid=((labels>=0)&(labels<=1)).all(axis=1)\n    p=p.loc[valid].copy(); labels=labels[valid]\n    rows=[]\n    for j,capacity in enumerate(capacities):\n        p[\'alternate_reference\']=labels[:,j]\n        for (kind,seed,rid),g in p.groupby([\'model\',\'seed\',\'record_id\']):\n            e=(g.SOC_estimate-g.alternate_reference).to_numpy()*100\n            rows.append(dict(model=kind,seed=seed,record_id=rid,capacity_scenario_Ah=capacity,\n                n_common_targets=len(g),MAE_pp=float(np.abs(e).mean()),RMSE_pp=float(np.sqrt((e**2).mean()))))\n    frame=pd.DataFrame(rows)\n    frame.to_csv(folder/\'reference_sensitivity_per_record.csv\',index=False)\n    macro=frame.groupby([\'model\',\'seed\',\'capacity_scenario_Ah\'],as_index=False)[[\'MAE_pp\',\'RMSE_pp\']].mean()\n    macro.to_csv(folder/\'reference_sensitivity_macro_per_seed.csv\',index=False)\n    summary=macro.groupby([\'model\',\'capacity_scenario_Ah\'])[[\'MAE_pp\',\'RMSE_pp\']].agg([\'mean\',\'std\'])\n    summary.to_csv(folder/\'reference_sensitivity_summary.csv\')\n    note=dict(stage=\'exploratory post hoc analysis motivated by data audit; not preregistered\',\n        calibration_capacity_Ah=qref,capacity_scenarios_Ah=capacities,\n        interpretation=\'Scenarios change the reference only. They are neither independent SOC truth nor measured capacity uncertainty. No model is retrained, selected or tuned.\',\n        timestamp_policy=\'same common-domain target timestamps for all three capacities, models and seeds\',\n        prediction_rows_before=len(valid),prediction_rows_after=int(valid.sum()),\n        distinct_common_targets=len(p[[\'record_id\',\'t_s\']].drop_duplicates()))\n    (folder/\'reference_sensitivity_notes.json\').write_text(json.dumps(note,indent=2))\n    print(summary.to_string())\n\nif __name__==\'__main__\':\n    main()\n', 'scripts/build_manuscript.py': '"""Build a review draft from completed, audited experiments; never invent results."""\nfrom pathlib import Path\nimport json, re\nimport numpy as np\nimport pandas as pd\nROOT=Path(__file__).resolve().parents[1]\n\ndef main():\n    folder=ROOT/\'results/full\'\n    m=pd.read_csv(folder/\'metrics_per_record.csv\')\n    test=m[m.split==\'test\'].copy()\n    expected={(kind,seed) for kind in [\'MLP\',\'GRU\',\'LSTM\'] for seed in [11,22,33]}\n    if set(zip(test.model,test.seed))!=expected or not (test.groupby([\'model\',\'seed\']).size()==12).all():\n        raise ValueError(\'Main experiment incomplete; no final manuscript generated\')\n    run=json.loads((folder/\'run_info.json\').read_text())\n    audit=json.loads((ROOT/\'results/pipeline_validation.json\').read_text())\n    inv=pd.read_csv(ROOT/\'results/data_inventory.csv\')\n    macro=test.groupby([\'model\',\'seed\'],as_index=False)[[\'MAE_pp\',\'RMSE_pp\']].mean()\n    stats=macro.groupby(\'model\')[[\'MAE_pp\',\'RMSE_pp\']].agg([\'mean\',\'std\'])\n    best=stats[\'RMSE_pp\'][\'mean\'].idxmin()\n    def number(x): return f\'{x:.3f}\'\n    def pm(kind,metric):\n        return f"{stats.loc[kind,(metric,\'mean\')]:.3f} ± {stats.loc[kind,(metric,\'std\')]:.3f}"\n    table=[\'| Model | MAE makro (pp) | RMSE makro (pp) | Kesalahan absolut terbesar (pp) | Parameter |\',\n           \'|---|---:|---:|---:|---:|\']\n    compute=[\'| Model | Waktu pelatihan termasuk validasi (s) | Throughput batch CPU (ms/jendela) |\',\n             \'|---|---:|---:|\']\n    for kind in [\'MLP\',\'GRU\',\'LSTM\']:\n        g=test[test.model==kind]\n        table.append(f"| {kind} | {pm(kind,\'MAE_pp\')} | {pm(kind,\'RMSE_pp\')} | {g.max_abs_error_pp.max():.3f} | {int(g.parameters.iloc[0]):,} |")\n        per_seed=g.drop_duplicates(\'seed\')\n        compute.append(f"| {kind} | {per_seed.training_seconds.mean():.2f} ± {per_seed.training_seconds.std():.2f} | {per_seed.batch_inference_ms_per_window.median():.4f} |")\n    merged=test.merge(inv[[\'record_id\',\'temperature_c\',\'profile\']],on=\'record_id\',validate=\'many_to_one\')\n    bytemp=merged.groupby([\'model\',\'seed\',\'temperature_c\'],as_index=False).RMSE_pp.mean()\n    tstats=bytemp.groupby([\'model\',\'temperature_c\']).RMSE_pp.agg([\'mean\',\'std\'])\n    temp_table=[\'| Model | 0 °C | 25 °C | 45 °C |\',\'|---|---:|---:|---:|\']\n    for kind in [\'MLP\',\'GRU\',\'LSTM\']:\n        temp_table.append(\'| \'+kind+\' | \'+\' | \'.join(f"{tstats.loc[(kind,t),\'mean\']:.3f} ± {tstats.loc[(kind,t),\'std\']:.3f}" for t in [0,25,45])+\' |\')\n    span=inv.groupby(\'temperature_c\').agg(records=(\'record_id\',\'count\'),samples=(\'n_samples\',\'sum\'),outside=(\'outside_0_1_samples\',\'sum\'))\n    audit_table=[\'| Suhu lingkungan | Rekaman | Sampel 1 detik | Acuan di luar 0–100% |\',\'|---|---:|---:|---:|\']\n    for t,g in span.iterrows():\n        audit_table.append(f"| {t} °C | {int(g.records):,} | {int(g.samples):,} | {int(g.outside):,} |")\n    results_text=\'; \'.join(f"{kind} {pm(kind,\'RMSE_pp\')} pp" for kind in [\'MLP\',\'GRU\',\'LSTM\'])\n    sensitivity=pd.read_csv(folder/\'reference_sensitivity_macro_per_seed.csv\')\n    sensitivity_stats=sensitivity.groupby([\'capacity_scenario_Ah\',\'model\']).RMSE_pp.agg([\'mean\',\'std\'])\n    sensitivity_notes=json.loads((folder/\'reference_sensitivity_notes.json\').read_text())\n    sensitivity_table=[\'| Kapasitas skenario (Ah) | MLP | GRU | LSTM |\',\'|---|---:|---:|---:|\']\n    for q in sorted(sensitivity.capacity_scenario_Ah.unique()):\n        sensitivity_table.append(f\'| {q:.5f} | \'+\' | \'.join(f"{sensitivity_stats.loc[(q,k),\'mean\']:.3f} ± {sensitivity_stats.loc[(q,k),\'std\']:.3f}" for k in [\'MLP\',\'GRU\',\'LSTM\'])+\' |\')\n    scenario_winners=[]\n    for q in sorted(sensitivity.capacity_scenario_Ah.unique()):\n        winner=sensitivity_stats.loc[q][\'mean\'].idxmin()\n        scenario_winners.append(f\'{q:.5f} Ah: {winner}\')\n    recurrent_difference=abs(stats.loc[\'GRU\',(\'RMSE_pp\',\'mean\')]-stats.loc[\'LSTM\',(\'RMSE_pp\',\'mean\')])\n    manuscript=f\'\'\'# Evaluasi Estimasi Muatan Baterai Litium-Ion pada Variasi Profil Beban dan Suhu\n\n**Evaluation of Lithium-Ion Battery State-of-Charge Estimation across Load Profiles and Ambient Temperatures**\n\nNama penulis, afiliasi, dan penulis korespondensi: **belum diisi**. Hapus bagian ini dari berkas review anonim sesuai template jurnal.\n\n> Status: draf penelitian komputasional untuk peninjauan penulis. Angka berasal dari pelatihan nyata pada dataset CALCE, bukan contoh atau hasil SOH terdahulu. Belum diterapkan ke template Mestro dan belum dinyatakan siap dikirim. NASA tidak digunakan dalam hasil berikut.\n\n## Abstrak\n\nEstimasi muatan baterai perlu diuji pada profil beban yang berbeda dari data pelatihan dengan definisi acuan yang dapat ditelusuri. Penelitian ini membandingkan multilayer perceptron, gated recurrent unit, dan long short-term memory pada 24 rekaman satu sel litium-ion CALCE, meliputi empat profil beban, tiga suhu lingkungan, dan dua kondisi muatan awal nominal. Acuan muatan direkonstruksi dari pencatatan kapasitas penguji menggunakan kapasitas kalibrasi terpisah sebesar {audit[\'Q_reference_Ah\']:.5f} Ah. Audit menemukan {audit[\'outside_reference_range_samples\']:,} dari {audit[\'raw_reference_samples\']:,} sampel acuan berada di luar domain 0–100%; label tersebut dipertahankan untuk audit dan jendela terkait dikeluarkan tanpa clipping. Profil DST dipakai untuk pelatihan, FUDS untuk validasi, serta US06 dan BJDST untuk pengujian. Semua model memakai konteks kausal 60 detik dan dijalankan pada tiga seed. RMSE makro pengujian diperoleh sebagai berikut: {results_text}. {best} memiliki rerata RMSE terendah pada konfigurasi yang diuji. Temuan dibatasi pada satu sel, suhu yang telah diwakili dalam pelatihan, dan acuan berbasis pencatatan muatan. Hasil menunjukkan perlunya menilai kesalahan estimasi bersama keterbatasan acuan dan kebutuhan komputasi.\n\n**Kata kunci:** estimasi muatan baterai; profil beban dinamis; jaringan saraf; suhu lingkungan; audit data.\n\n## Abstract\n\nBattery state-of-charge estimation requires evaluation on load profiles distinct from the training data and a traceable reference definition. This study compares a multilayer perceptron, a gated recurrent unit, and a long short-term memory network using 24 records from one CALCE lithium-ion cell, covering four load profiles, three ambient temperatures, and two nominal initial charge levels. Reference state of charge is reconstructed from tester capacity counters using a separate initial-capacity calibration of {audit[\'Q_reference_Ah\']:.5f} Ah. The audit identifies {audit[\'outside_reference_range_samples\']:,} of {audit[\'raw_reference_samples\']:,} samples outside the 0–100% reference domain; original labels are retained and affected windows are excluded without clipping. DST is used for training, FUDS for validation, and US06 and BJDST for testing. All models use a causal 60-second context and three random seeds. Test macro-RMSE values are {results_text}. {best} yields the lowest mean macro-RMSE for the evaluated configuration. Findings are limited to one cell, temperatures represented during training, and a charge-counter-based reference. The results emphasize interpreting estimation errors together with reference limitations and computational requirements.\n\n**Keywords:** state-of-charge estimation; dynamic load profile; neural network; ambient temperature; data audit.\n\n## 1. Pendahuluan\n\nState of charge (SOC) menyatakan tingkat muatan baterai relatif terhadap kapasitas acuan yang ditetapkan. Dalam pengembangan estimator, pemilihan kapasitas acuan berhubungan langsung dengan label target dan interpretasi kesalahan. Zheng et al. menunjukkan bahwa hubungan tegangan rangkaian terbuka dan SOC serta pengaruh suhu perlu diperhatikan ketika mengembangkan estimator daring [1]. Oleh sebab itu, kesalahan numerik yang rendah terhadap suatu label harus dibaca bersama cara label tersebut diperoleh.\n\nPendekatan pembelajaran mesin memungkinkan pemetaan riwayat sinyal listrik ke SOC. Chemali et al. telah menerapkan long short-term memory (LSTM) untuk estimasi SOC baterai [2]. Literatur yang lebih baru juga mengembangkan kombinasi temporal convolutional network dan unscented Kalman filter pada data CALCE [3], serta model TRNN-CTA untuk mengekstraksi informasi temporal [4]. Batool et al. mengevaluasi kerangka fuzzy–LSTM dengan profil beban CALCE pada beberapa suhu [5]. Dengan demikian, penggunaan jaringan rekuren, CALCE, atau pengujian lintas profil tidak dinyatakan sebagai kebaruan tersendiri dalam penelitian ini.\n\nPertanyaan yang dikaji adalah bagaimana tiga konfigurasi estimator relatif sederhana berbeda ketika diuji pada profil beban yang tidak dipakai untuk pelatihan, dengan konteks masukan dan aturan pemisahan yang sama. Multilayer perceptron (MLP) dipakai sebagai pembanding tanpa keadaan rekuren. Gated recurrent unit (GRU) dan LSTM mewakili dua arsitektur rekuren yang telah dikenal [6], [7]. Pembandingan juga mempertimbangkan jumlah parameter serta waktu komputasi, sehingga kesimpulan tidak hanya didasarkan pada satu skor ketepatan.\n\nPenelitian ini menyajikan audit acuan SOC, pemisahan rekaman sebelum pembentukan jendela, evaluasi per rekaman pada tiga suhu lingkungan, dan pelaporan variasi antarseed. Kontribusinya bersifat empiris dan reproduktif pada konfigurasi yang dinyatakan. Penelitian tidak mengusulkan arsitektur baru, tidak menggunakan hasil SOH sebagai hasil SOC, dan tidak mengklaim generalisasi lintas sel atau implementasi pada sistem manajemen baterai.\n\n## 2. Metode Penelitian\n\n### 2.1 Dataset dan unit analisis\n\nData diunduh dari repositori resmi CALCE [8]. Subset yang digunakan adalah sel INR 18650-20R dengan penamaan berkas SP20-2. Dua belas arsip dinamis memuat profil DST, FUDS, US06, dan BJDST pada suhu lingkungan 0, 25, dan 45 °C. Setiap kombinasi profil–suhu mempunyai dua rekaman dengan muatan awal nominal 50% dan 80%. Satu arsip tambahan menyediakan pengujian kapasitas awal. Unit analisis adalah rekaman pengujian berulang pada **satu sel**, bukan 24 baterai independen. URL asal, nama anggota arsip, dan SHA-256 disimpan dalam manifest.\n\nBagian profil dinamis dipisahkan dari pengondisian pengisian/pengosongan awal. Urutan waktu dan penghitung kapasitas diperiksa. Sinyal dibentuk pada grid satu detik dengan mengambil observasi terakhir yang sudah tersedia. Cara ini tidak memakai interpolasi terpusat atau sampel masa depan. Sampel dengan umur observasi lebih dari lima detik tidak dipakai untuk membentuk jendela.\n\n### 2.2 Rekonstruksi acuan dan audit domain\n\nKapasitas acuan diperoleh dari pengosongan penuh pada berkas pengujian kapasitas awal terpisah, yaitu Qref = {audit[\'Q_reference_Ah\']:.5f} Ah. Nilai pencatat lengkap dipertahankan dalam kode untuk replikasi, tanpa menyatakan ketelitian alat hingga seluruh digit tersebut. Titik akhir pengisian CCCV dianggap sebagai keadaan penuh. Dengan Qc dan Qd sebagai penghitung kumulatif kapasitas pengisian dan pengosongan, acuan dinyatakan oleh:\n\nSOCref(t) = 1 + [(Qc(t) − Qc,full) − (Qd(t) − Qd,full)] / Qref. (1)\n\nArus positif pada sumber berarti pengisian. Efisiensi coulombic diasumsikan satu. SOCref merupakan acuan hasil rekonstruksi pencatatan muatan, bukan pengukuran SOC independen. Kapasitas awal yang tetap juga tidak menyatakan kapasitas efektif identik pada semua suhu atau profil.\n\nAudit menemukan {audit[\'outside_reference_range_samples\']:,} sampel ({audit[\'outside_reference_range_percent\']:.3f}%) di luar domain SOC 0–100%, terutama pada akhir pengosongan. Nilai mentah tetap disimpan. Jendela yang memuat label di luar domain dikeluarkan secara seragam dari semua model tanpa memotong label ke nol atau satu. Pengecualian ini membatasi kesimpulan pada domain acuan yang diterima; penelitian tidak mengklaim ketepatan seluruh lintasan sampai cutoff. Ketidakpastian kapasitas acuan belum dikuantifikasi sebagai interval SOC fisik. Analisis Movassagh et al. menegaskan bahwa ketidakpastian kapasitas, pengukuran arus, integrasi, dan waktu dapat memengaruhi hasil coulomb counting [10]; audit di sini bukan klaim penemuan baru mengenai sumber ketidakpastian tersebut.\n\n**Tabel 1. Audit sampel acuan sebelum pembentukan jendela.**\n\n{chr(10).join(audit_table)}\n\n![Audit acuan SOC](results/full/figures/reference_audit.png)\n\n**Gambar 1.** Rekonstruksi acuan pada DST dengan kondisi awal nominal 80%. Nilai negatif ditampilkan untuk memperlihatkan keterbatasan kapasitas acuan tetap; nilai tersebut tidak disembunyikan melalui clipping.\n\n### 2.3 Masukan dan pembagian data\n\nVektor masukan terdiri atas tegangan terminal, arus, dan setelan suhu lingkungan ruang uji. Suhu tersebut tidak diperlakukan sebagai pembacaan suhu sel per detik. Penghitung kapasitas, indeks waktu, label SOC, dan data masa depan tidak digunakan sebagai fitur. Normalisasi berupa pengurangan rerata dan pembagian simpangan baku yang dihitung hanya dari sampel latih yang memenuhi domain.\n\nProfil DST pada ketiga suhu digunakan untuk pelatihan. FUDS pada ketiga suhu digunakan untuk validasi dan early stopping. US06 serta BJDST pada ketiga suhu digunakan untuk pengujian. Rekaman dipisahkan sebelum pembentukan jendela sehingga jendela tumpang tindih tidak berpindah antara latih dan uji. Pemisahan ini menilai profil beban yang ditahan, sementara suhu uji telah diwakili pada pelatihan.\n\nSetiap jendela memuat 60 sampel hingga waktu t dan menghasilkan SOC pada waktu t. Keadaan rekuren diinisialisasi ulang untuk setiap jendela. Langkah pembentukan jendela latih adalah lima sampel, sedangkan validasi dan uji satu sampel. Seluruh model dievaluasi pada titik target identik. Setelah pemeriksaan jendela, terdapat {run[\'train_windows\']:,} jendela latih, {run[\'validation_windows\']:,} jendela validasi, dan {run[\'test_windows\']:,} jendela uji. Jendela yang bertumpang tindih tetap berkorelasi dan tidak dihitung sebagai replikasi sel independen.\n\n### 2.4 Model dan pelatihan\n\nMLP menerima 180 nilai hasil perataan jendela 60 × 3, diikuti satu lapisan tersembunyi 32 unit dengan aktivasi tanh. GRU dan LSTM menggunakan satu lapisan rekuren searah dengan 32 unit tersembunyi. Keluaran keadaan terakhir diteruskan ke lapisan linear dan sigmoid untuk membatasi estimasi pada 0–1. Arsitektur mengikuti implementasi standar PyTorch; tidak ada modifikasi gerbang yang diusulkan.\n\nSemua konfigurasi menggunakan mean squared error, Adam dengan learning rate 0,001 [9], batch 256, dan pemotongan norma gradien maksimum satu. Pelatihan dibatasi 50 epoch dengan patience delapan berdasarkan MSE validasi gabungan. Bobot pada MSE validasi minimum dipakai untuk evaluasi. Seed 11, 22, dan 33 digunakan untuk memeriksa variasi inisialisasi dan pengacakan urutan batch. Pilot lima epoch pada 25 °C memakai latih/validasi saja; skor pilot tidak dicampurkan dengan pengujian utama.\n\nJumlah unit tersembunyi dan konteks disamakan, sedangkan jumlah parameter tidak identik. Oleh karena itu, pembandingan merupakan evaluasi konfigurasi yang dinyatakan dan bukan pembuktian arsitektur dengan kapasitas model persis sama. Pemilihan model atau parameter tidak dilakukan berdasarkan skor uji. Semua arsitektur tetap dilaporkan.\n\n### 2.5 Metrik dan komputasi\n\nMAE dan RMSE dihitung dalam poin persentase SOC (pp) untuk setiap rekaman. Kesalahan adalah 100 × (SOCestimasi − SOCref). Rerata makro memberi bobot sama pada 12 rekaman uji. Rerata dan simpangan baku kemudian dihitung atas tiga seed. Kesalahan absolut terbesar diperoleh dari seluruh sampel uji pada semua seed untuk masing-masing model. Simpangan baku antarseed tidak ditafsirkan sebagai interval kepercayaan populasi baterai.\n\nJumlah parameter, waktu pelatihan termasuk evaluasi validasi tiap epoch, dan throughput batch CPU dilaporkan. Pengukuran inferensi dilakukan setelah pemanasan dengan batch 256 dan dua thread CPU. Satuan ms/jendela merupakan waktu total inferensi batch dibagi jumlah jendela, bukan latensi pemrosesan satu sampel di ESP32. Versi runtime eksperimen adalah PyTorch {run[\'torch\']} dan Python {run[\'python\']}; konfigurasi serta versi dependensi disimpan bersama kode.\n\n## 3. Hasil dan Pembahasan\n\n### 3.1 Ketepatan pada profil beban yang ditahan\n\n**Tabel 2. Ketepatan pengujian utama.** Rerata ± simpangan baku dari tiga seed; MAE/RMSE merupakan rerata makro per rekaman. Kesalahan maksimum diambil dari seluruh rekaman dan seed.\n\n{chr(10).join(table)}\n\nPada konfigurasi ini, {best} memiliki rerata RMSE makro terendah. Selisih rerata RMSE GRU dan LSTM adalah {recurrent_difference:.3f} pp. Perbedaan tersebut bersifat deskriptif untuk profil US06/BJDST dan definisi acuan yang dipakai. Penelitian tidak melakukan uji signifikansi berbasis replikasi sel independen. Dengan demikian, selisih skor tidak dinyatakan sebagai keunggulan universal atau bukti ketepatan SOC fisik pada seluruh baterai.\n\n![Perbandingan RMSE](results/full/figures/rmse_comparison.png)\n\n**Gambar 2.** RMSE makro uji; error bar menunjukkan variasi antarseed dan bukan ketidakpastian populasi sel.\n\n![Lintasan estimasi SOC](results/full/figures/soc_trace.png)\n\n**Gambar 3.** Contoh lintasan US06, 25 °C, kondisi awal nominal 80%, seed 11. Rekaman dan seed untuk visualisasi ditetapkan dalam kode sebelum eksperimen utama selesai dan bukan dipilih berdasarkan skor terbaik.\n\n### 3.2 Perbedaan menurut suhu lingkungan\n\n**Tabel 3. RMSE makro per suhu lingkungan (pp).** Pada setiap suhu, empat rekaman US06/BJDST berbobot sama; rerata ± simpangan baku dihitung antarseed.\n\n{chr(10).join(temp_table)}\n\n![RMSE menurut suhu](results/full/figures/rmse_by_temperature.png)\n\n**Gambar 4.** Perbedaan skor pada suhu yang telah diwakili dalam pelatihan.\n\nPemisahan menurut suhu membantu menunjukkan apakah satu skor gabungan menyembunyikan kondisi yang lebih sulit. Namun, ketiga suhu masuk data latih, sehingga hasil ini tidak membuktikan generalisasi ke suhu baru. Efek suhu pada sinyal listrik dan perbedaan acuan kapasitas juga tidak dipisahkan secara kausal dalam rancangan ini. Pada suhu rendah, penghentian pengosongan dapat terjadi ketika acuan berbasis kapasitas tetap masih positif; pada sebagian rekaman suhu lebih tinggi, rekonstruksi melewati nol. Perbedaan ini memperlihatkan bahwa definisi SOC, kapasitas yang digunakan, dan batas pengosongan harus dipertimbangkan bersamaan.\n\n### 3.3 Kebutuhan komputasi\n\n**Tabel 4. Pengukuran komputasi pada CPU pengujian.** Waktu pelatihan adalah rerata ± simpangan baku antarseed; throughput inferensi adalah median antarseed pada seluruh jendela uji.\n\n{chr(10).join(compute)}\n\nAngka waktu hanya berlaku pada perangkat dan implementasi pengujian. Jumlah parameter dapat digunakan untuk membandingkan ukuran model, tetapi belum membuktikan kebutuhan RAM, latensi, konsumsi daya, atau kompatibilitas ESP32. Implementasi embedded memerlukan pengukuran tambahan. Pelatihan termasuk validasi tiap epoch, sehingga waktu total juga dipengaruhi banyaknya epoch terpilih dan biaya evaluasi.\n\n### 3.4 Sensitivitas terhadap definisi kapasitas acuan\n\nPemeriksaan eksploratif tambahan mengubah kapasitas acuan label menjadi 2,00 Ah, kapasitas kalibrasi, dan 2,06 Ah tanpa melatih ulang model. Ketiga nilai merupakan skenario, bukan toleransi kapasitas terukur atau batas ketidakpastian SOC benar. Agar pengaruh pergantian label tidak tercampur dengan pergantian sampel, seluruh skenario memakai {sensitivity_notes[\'distinct_common_targets\']:,} titik target yang sama dan memenuhi domain acuan untuk ketiganya. Angka pada tabel ini tidak boleh dibandingkan sebagai peningkatan terhadap Tabel 2 tanpa memperhatikan perbedaan cakupan titik target.\n\n**Tabel 5. RMSE makro pada skenario definisi acuan (pp).** Rerata ± simpangan baku antarseed. Ini sensitivitas skor terhadap label, bukan akurasi terverifikasi terhadap kapasitas alternatif.\n\n{chr(10).join(sensitivity_table)}\n\nModel dengan rerata RMSE terendah pada setiap skenario adalah {\'; \'.join(scenario_winners)}. Pemeriksaan ini menunjukkan bagaimana evaluasi numerik dan urutan model dapat bergantung pada asumsi kapasitas yang digunakan dalam label. Tidak ada kapasitas skenario yang dipilih untuk meningkatkan skor. Model, bobot, dan konfigurasi tetap sama. Analisis ini bersifat post hoc setelah audit data dan tidak dinyatakan sebagai eksperimen yang dipreregistrasikan.\n\n### 3.5 Posisi terhadap literatur dan keterbatasan\n\nLSTM untuk SOC telah diperkenalkan dalam studi terdahulu [2], dan penelitian terbaru mengembangkan model temporal atau gabungan dengan filter [3]–[5]. Skor penelitian ini tidak dibandingkan sebagai peringkat langsung terhadap angka terbitan tersebut karena sel, rekaman, acuan SOC, preprocessing, dan protokol dapat berbeda. Pembandingan yang dapat dipertanggungjawabkan di sini adalah antarimplementasi yang dilatih dan dievaluasi ulang dalam pipeline yang sama.\n\nKeterbatasan paling mendasar adalah penggunaan acuan rekonstruksi. Kesalahan terhadap penghitung muatan bukan pengukuran kesalahan terhadap SOC independen. Kapasitas acuan tetap menghasilkan label di luar domain pada sebagian akhir rekaman; pengecualian seragam mencegah clipping menyembunyikan persoalan, tetapi tetap membatasi cakupan evaluasi. Penelitian lanjutan perlu menilai sensitivitas terhadap kapasitas acuan, memperoleh kalibrasi per kondisi yang dapat ditelusuri, dan memperluas pengujian pada sel lain.\n\nKeterbatasan lain meliputi satu sel, tiga seed, tidak adanya eksperimen hardware, dan tidak adanya uji suhu yang sepenuhnya ditahan dalam hasil utama. Pembandingan dengan estimator fisik seperti EKF/UKF juga belum dilakukan, sehingga hasil tidak membuktikan keunggulan terhadap keluarga metode tersebut. Data NASA tidak dimasukkan karena audit acuan dan kesetaraan protokolnya belum diselesaikan. Temuan ini tidak boleh dipakai untuk mengklaim performa lintas NASA–CALCE, prediksi umur baterai, atau kesiapan BMS.\n\n## 4. Kesimpulan\n\nPenelitian membandingkan MLP, GRU, dan LSTM pada 24 rekaman dinamis satu sel CALCE melalui pemisahan profil sebelum pembuatan jendela. RMSE makro pengujian adalah {results_text}. {best} memperoleh rerata RMSE terendah pada konfigurasi yang dinyatakan. Audit juga mengidentifikasi {audit[\'outside_reference_range_samples\']:,} sampel acuan di luar domain akibat keterbatasan penggunaan kapasitas tetap; label dipertahankan dan jendela terkait dikeluarkan tanpa clipping. Hasil mendukung pembandingan empiris terhadap acuan rekonstruksi, dengan batas pada satu sel dan suhu yang telah diwakili selama pelatihan. Validasi acuan per kondisi, pengujian sel independen, dan pengukuran embedded diperlukan sebelum menyatakan penerapan pada sistem manajemen baterai.\n\n## Ketersediaan Data dan Kode\n\nData asli tersedia pada repositori CALCE [8]. Kode pengunduhan, audit, preprocessing, pelatihan, konfigurasi, hash sumber, dan prediksi disertakan dalam paket penelitian. URL repositori publik kode belum ditetapkan. Tidak ada klaim bahwa paket telah diarsipkan pada repositori publik atau memiliki DOI.\n\n## Penggunaan AI dalam Penyusunan Draf\n\nCodex dari OpenAI digunakan untuk membantu penelusuran literatur, penyusunan kode, audit komputasional, visualisasi, dan penulisan draf. Pelatihan model dijalankan pada data sumber yang diunduh; hasil bukan angka ilustratif. Penulis manusia perlu memeriksa, merevisi, dan menyetujui naskah serta mengambil tanggung jawab ilmiah sebelum pengiriman. Pernyataan ini tidak menyatakan persetujuan tersebut sudah diberikan.\n\n## Daftar Pustaka\n\n[1] F. Zheng, Y. Xing, J. Jiang, B. Sun, J. Kim, dan M. Pecht, “Influence of different open circuit voltage tests on state of charge online estimation for lithium-ion batteries,” Applied Energy, vol. 183, pp. 513–525, 2016. https://doi.org/10.1016/j.apenergy.2016.09.010.\n\n[2] E. Chemali, P. J. Kollmeyer, M. Preindl, R. Ahmed, dan A. Emadi, “Long Short-Term Memory Networks for Accurate State-of-Charge Estimation of Li-ion Batteries,” IEEE Transactions on Industrial Electronics, vol. 65, no. 8, pp. 6730–6739, 2018. https://doi.org/10.1109/TIE.2017.2787586.\n\n[3] H. Lin, X. Chen, L. Zhang, Z. Bao, dan S. Tang, “Estimation of battery SOC using a combined approach of temporal convolutional networks and Unscented Kalman Filter,” PLOS One, vol. 21, no. 6, e0350331, 2026. https://doi.org/10.1371/journal.pone.0350331.\n\n[4] Y. Wu, J. Zhuang, L. Liu, F. Zeng, P. Li, dan X. Cheng, “Research on Estimation of State of Charge for Lithium-Ion Batteries Based on TRNN-CTA,” Applied Sciences, vol. 15, no. 17, 9653, 2025. https://doi.org/10.3390/app15179653.\n\n[5] D. Batool, F. Majeed, S. Oh, J. Oh, dan J. Kim, “Hybrid Fuzzy Long Short-Term Memory Framework-Based Accurate State-Of-Charge Estimation for Future-Oriented Battery Management in Vehicle-To-Grid System,” International Journal of Energy Research, 5529603, 2026. https://doi.org/10.1155/er/5529603.\n\n[6] K. Cho et al., “Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation,” EMNLP, 2014. https://doi.org/10.48550/arXiv.1406.1078.\n\n[7] S. Hochreiter dan J. Schmidhuber, “Long Short-Term Memory,” Neural Computation, vol. 9, no. 8, pp. 1735–1780, 1997. https://doi.org/10.1162/neco.1997.9.8.1735.\n\n[8] CALCE, University of Maryland, “Battery Data,” diakses 22 Oktober 2026. https://calce.umd.edu/battery-data.\n\n[9] D. P. Kingma dan J. Ba, “Adam: A Method for Stochastic Optimization,” ICLR, 2015. https://doi.org/10.48550/arXiv.1412.6980.\n\n[10] K. Movassagh, A. Raihan, B. Balasingam, dan K. Pattipati, “A Critical Look at Coulomb Counting Approach for State of Charge Estimation in Batteries,” Energies, vol. 14, no. 14, 4074, 2021. https://doi.org/10.3390/en14144074.\n\'\'\'\n    def url_link(match):\n        raw=match.group(0); url=raw.rstrip(\'.\')\n        return f\'[Sumber]({url})\'+raw[len(url):]\n    manuscript=re.sub(r\'(?<!\\()https://\\S+\',url_link,manuscript)\n    (ROOT/\'DRAF_ARTIKEL_MESTRO_SOC.md\').write_text(manuscript,encoding=\'utf-8\')\n    stats.to_csv(folder/\'manuscript_summary.csv\')\n    tstats.to_csv(folder/\'manuscript_temperature_summary.csv\')\n    print(\'Draft generated from complete experiment; best mean macro RMSE:\',best)\n\nif __name__==\'__main__\':\n    main()\n', 'protocol_v1.json': '{\n  "stage": "development protocol; no claim of preregistration",\n  "target": "current SOC estimation (not future SOC forecasting, SOH or RUL)",\n  "primary_dataset": "CALCE INR 18650-20R SP20-2 dynamic profiles",\n  "nasa_status": "conditional extension; excluded until suitable SOC reference and inputs are audited",\n  "inputs": ["voltage_V", "current_A_positive_charge", "ambient_chamber_temperature_C"],\n  "forbidden_inputs": ["charge_capacity", "discharge_capacity", "reference_SOC", "test_time", "future_samples"],\n  "reference": "tester charge counters anchored at full CCCV; Qref from separate initial-capacity run; not independently measured ground truth",\n  "reference_quality_gate": "exclude windows containing reconstructed SOC outside [0,1]; preserve raw labels and report exclusion counts; do not clip targets; no claim about complete cutoff-to-cutoff SOC accuracy",\n  "reference_limitation": "fixed initial calibration capacity gives negative end-of-discharge references in some 25/45 C records; this is a reference-definition limitation, not proof of sensor errors; temperature-dependent capacity sensitivity requires further validation",\n  "models": ["MLP", "GRU", "LSTM"],\n  "train_profiles": ["DST"],\n  "validation_profiles": ["FUDS"],\n  "test_profiles": ["US06", "BJDST"],\n  "primary_temperatures_C": [0, 25, 45],\n  "cross_temperature_extension": "train DST at 0/45 C, validate FUDS at 0/45 C, test US06/BJDST at held-out 25 C; separate training run",\n  "window_samples": 60,\n  "sampling_seconds": 1,\n  "train_stride": 5,\n  "evaluation_stride": 1,\n  "hidden_units": 32,\n  "epochs_max": 50,\n  "early_stopping_patience": 8,\n  "batch_size": 256,\n  "learning_rate": 0.001,\n  "seeds": [11, 22, 33],\n  "normalization": "fit pooled training records only; reused for validation and testing",\n  "metrics": ["MAE_pp", "RMSE_pp", "max_abs_error_pp", "parameters", "training_seconds", "batch_inference_ms_per_window"],\n  "aggregation": "equal-weight record macro-average, with per-record metrics retained; same target times for every model",\n  "tuning_policy": "validation only; no model chosen or tuned by test errors",\n  "pilot": "25 C, one seed, five epochs; training and validation only; no test metrics",\n  "claims_boundary": "one cell; ambient temperature not cell temperature; no hardware/BMS validation; reconstructed reference uncertainty"\n}\n', 'requirements.txt': 'numpy>=1.26,<3\npandas>=2.2,<4\nopenpyxl>=3.1,<4\nxlrd==2.0.2\nmatplotlib>=3.8,<4\ntorch>=2.4,<3\n'}
for name, content in embedded.items():
    path=ROOT/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(content,encoding='utf-8')
def run(script,*args):
    subprocess.check_call([sys.executable,str(ROOT/'scripts'/script),*args],cwd=ROOT)
print(ROOT)


## 1. Unduh dan audit data asli

Sumber: https://calce.umd.edu/battery-data. Unduhan disertai SHA-256; berkas tidak ditimpa jika telah tersedia.

In [ ]:
run('fetch_data.py','--temperatures','0','25','45')
run('prepare_data.py')
run('validate_pipeline.py')


## 2. Pilot: pelatihan dan validasi saja

Satu seed, lima epoch, 25 C. Tidak mengevaluasi US06/BJDST.

In [ ]:
run('train.py','--mode','pilot')
run('plot_results.py','--mode','pilot')


## 3. Eksperimen utama

DST latih, FUDS validasi, US06/BJDST uji. Ketiga suhu ada dalam data latih. Tiga seed; maksimal 50 epoch; early stopping. Jangan memilih parameter berdasarkan hasil uji.

In [ ]:
run('train.py','--mode','full')
run('audit_results.py')
run('plot_results.py','--mode','full')
run('reference_sensitivity.py')
run('build_manuscript.py')


In [ ]:
import pandas as pd
display(pd.read_csv(ROOT/'results/full/summary.csv',header=[0,1],index_col=[0,1]))
from IPython.display import Image, display
display(Image(filename=str(ROOT/'results/full/figures/rmse_comparison.png')))
display(Image(filename=str(ROOT/'results/full/figures/soc_trace.png')))


## 4. Ekstensi opsional: suhu 25 C ditahan

Ini pelatihan baru, bukan evaluasi ulang model utama. DST/FUDS 0 dan 45 C dipakai untuk latih/validasi. US06/BJDST 25 C menjadi uji. Sel di bawah tidak perlu dijalankan untuk mereproduksi eksperimen utama.

In [ ]:
# Hapus komentar jika ingin menjalankan ekstensi.
# run('train.py','--mode','cross-temperature')
# run('plot_results.py','--mode','cross-temperature')


## 5. Ekspor hasil

Simpan konfigurasi, metrik, prediksi, dan model. Setiap angka naskah harus berasal dari keluaran aktual, bukan contoh.

In [ ]:
import shutil
export=shutil.make_archive(str(ROOT.parent/'SOC_CALCE_results'),'zip',ROOT/'results')
print(export)
try:
    from google.colab import files
    files.download(export)
except ImportError:
    pass
